In [2]:
import torch, torch.nn as nn, torchvision

In [3]:
from torchvision.models import resnet101, ResNet101_Weights
model = resnet101(weights=ResNet101_Weights.DEFAULT)

In [4]:
num_iterations = 1
xe = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)

# Basic example

In [5]:
desired_batch_size = 100
tolerated_batch_size = 20
accum_steps = desired_batch_size//tolerated_batch_size

for i in range(num_iterations):
    inputs = torch.randn(tolerated_batch_size, 3, 224, 224)
    labels = torch.randint(0, 100, (tolerated_batch_size,))
    loss = xe(model(inputs), labels)
    loss /= accum_steps
    loss.backward()

    if (i+1) % accum_steps == 0:
        optimizer.step()
        optimizer.zero_grad()

    print(f"Done with batch {i+1}")

Done with batch 1


# CNN with gradient accumulation

In [5]:
desired_batch_size = 100      # batch size logic
tolerated_batch_size = 20     # batch GPU có thể load
accum_steps = desired_batch_size // tolerated_batch_size
train_loader = DataLoader(dataset, batch_size=desired_batch_size, shuffle=True)

for i, (imgs, labels) in enumerate(train_loader):
    # chia batch lớn ra mini-batch nhỏ
    for j in range(accum_steps):
        start = j * tolerated_batch_size
        end = start + tolerated_batch_size

        inputs = imgs[start:end, :, :, :]
        mini_labels = labels[start:end]

        outputs = model(inputs)
        loss = xe(outputs, mini_labels)
        loss = loss / accum_steps   # scale loss
        loss.backward()             # accumulate gradient

    # step optimizer sau 1 batch lớn
    optimizer.step()
    optimizer.zero_grad()

    print(f"Done with effective batch {i+1}")